# 분류기 만들기

In [151]:
import pandas as pd
from matplotlib import rcParams
rcParams['font.family'] = 'Gulim'
rcParams['axes.unicode_minus'] = False

import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
df = pd.read_csv('./data/titanic.csv')

In [152]:
y_df = df['Survived']
x_df = df.drop('Survived', axis = 1)

In [153]:
from sklearn.preprocessing import LabelEncoder

# Null 처리 함수
def fillna(df):
    df['Age'].fillna(df['Age'].mean(), inplace=True)
    df['Cabin'].fillna('N', inplace=True)
    df['Embarked'].fillna('N', inplace=True)
    df['Fare'].fillna(0, inplace=True)
    return df

# 머신러닝 알고리즘에 불필요한 피처 제거
def drop_features(df):
    df.drop(['PassengerId', 'Name', 'Ticket'], axis=1, inplace=True)
    return df

# 레이블 인코딩 수행 함수
def format_features(df):
    df['Cabin'] = df['Cabin'].str[:1]
    features = ['Cabin', 'Sex', 'Embarked']
    for feature in features:
        le = LabelEncoder()
        le = le.fit(df[feature])
        df[feature] = le.transform(df[feature])
    return df

# 앞에서 설정한 데이터 전처리 함수 호출
def transform_features(df):
    df = fillna(df) 
    df = drop_features(df)
    df = format_features(df)
    return df

In [154]:
x_df1 = transform_features(x_df)

In [155]:
from sklearn.model_selection import train_test_split
# 데이터를 훈련용(train)과 테스트용(test)으로 나눠주는 함수
x_train, x_test, y_train, y_test = train_test_split(x_df1, y_df, test_size = 0.2, random_state = 10)

In [156]:
from sklearn.base import BaseEstimator
import numpy as np
class MyDummyClassifier(BaseEstimator):
    def fit(self, x, y):
        pass
    def predict(self, x):
        pred = np.zeros((x.shape[0],1))
        for i in range(x.shape[0]):
            if x['Sex'].iloc[i] == 1:
                pred[i] = 1
            else:
                pred[i] = 0
            return pred

In [157]:
myclf = MyDummyClassifier()
myclf.fit(x_train, y_train)
my_pred = myclf.predict(x_test)
from sklearn.metrics import accuracy_score
accuracy_score(y_test, my_pred)

0.6480446927374302

In [158]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test, my_pred)

array([[116,   1],
       [ 62,   0]])

In [159]:
from sklearn.metrics import precision_score, recall_score
precision_score(y_test, my_pred), recall_score(y_test, my_pred)

(np.float64(0.0), np.float64(0.0))

# 로지스틱 회귀, 랜덤포레스트, knn 정밀도와 재현율 비교하기

In [160]:
from sklearn.linear_model import LogisticRegression
x_train, x_test, y_train, y_test = train_test_split(x_df1, 
                 y_df, 
                 test_size= 0.2, 
                 random_state= 10)

In [161]:
model = LogisticRegression(random_state= 11)
model.fit(x_train, y_train)
pred = model.predict(x_test)

In [162]:
# 정밀도와 재현율 비교함수
def get_clf_eval(y_test, pred):
    confusion = confusion_matrix(y_test, pred)
    accuracy = accuracy_score(y_test, pred)
    precision = precision_score(y_test, pred)
    recall = recall_score(y_test,  pred)

    print(confusion)
    print('+' * 40)
    print(accuracy, precision, recall)

In [163]:
get_clf_eval(y_test, pred)

[[101  16]
 [ 15  47]]
++++++++++++++++++++++++++++++++++++++++
0.8268156424581006 0.746031746031746 0.7580645161290323


In [164]:
pred_proba = model.predict_proba(x_test)
pos_proba = pred_proba[:,1] # 양성클래스일 확률
threshold = 0.25 # 임계치

# 임계치보다 크면 1로 넣음
custom_proba = (pos_proba>=threshold).astype(int) 
custom_proba

array([0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1,
       1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1,
       1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0,
       1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1,
       0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1,
       1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0,
       0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0,
       0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1,
       0, 0, 1])

In [165]:
confusion_matrix(y_test, custom_proba)

array([[83, 34],
       [ 7, 55]])

In [166]:
get_clf_eval(y_test, custom_proba)

[[83 34]
 [ 7 55]]
++++++++++++++++++++++++++++++++++++++++
0.770949720670391 0.6179775280898876 0.8870967741935484


#  정밀도와 재현율의 변화

정밀도와 재현율의 불균형이 심할 때,   
혹은 비즈니스의 요구사항이 있을 때,      
임계치를 조정해야 한다          

임계치를 낮추면 정밀도는 낮아지고 재현율은 올라간다 

# 평가결과 확인하기

In [175]:
from sklearn.metrics import f1_score, classification_report
f1_score(y_test, pred)

np.float64(0.752)

In [177]:
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.87      0.86      0.87       117
           1       0.75      0.76      0.75        62

    accuracy                           0.83       179
   macro avg       0.81      0.81      0.81       179
weighted avg       0.83      0.83      0.83       179



In [ ]:
from sklearn.datasets import load_wine
wine = load_wine()